# fibonacci_collision_analysis

有限1次元Fibonacci-QECCで、任意消失と連続消失の性能差を巡回orbitのdifference supportから調べます。
既存の `fibonacci_seed_variation`、`fibonacci_recoverability_kl`、`fibonacci_finite_size_scaling`、
`fibonacci_logical_information_leakage` の保存済みコードと定義を照合しました。出典・SHA256は `reference/` にあります。

本番設定のNotebookと、小規模実行済みの `_example.ipynb` を同梱します。本番設定ではまずvalidation・計算量計画・benchmarkを表示します。
手動セルの `RUN_FULL_ANALYSIS=True` で本計算を行います。小規模例は仮説の一般的な証明ではありません。

## 1. 研究目的と検証する問い

- Q1: $d_{\rm cross}$ と任意消失訂正率に関連があるか。
- Q2: $\ell_{\rm cross,min}$ と連続消失訂正率に関連があるか。
- Q3–Q4: 任意消失に弱く連続消失に強いpairに、小weight・大spanのsupportが多いか。
- Q5: spectrum・multiplicity・期待collision数は訂正率予測の指標となるか。
- Q6: その指標を新しいseed pairの設計に使えるか。

「collisionが訂正失敗の全原因である」とは仮定しません。対角KL不一致も独立に計算します。


## 2–4. Collision・difference support・Hamming weight・cyclic span

siteは左から **0始まり**。$E$ は消失集合、$E^c$ は残存集合です。

$$D(x,y)=\{i:x_i\ne y_i\},\qquad x_{E^c}=y_{E^c}\iff D(x,y)\subseteq E.$$

$$w(x,y)=|D(x,y)|,\qquad \ell_{\rm cyc}(D)=\text{Dを含む最短cyclic intervalのsite数}.$$

空集合ではspan=0。非空のDを巡回順に並べ、隣接site間の距離の最大を$g_{\max}$とすると、
$\ell_{\rm cyc}=N-g_{\max}+1$です。端点も数えるので **+1** が必要です。
singletonは1、全siteならNです。N=20、D={5,6,8}なら4、D={2,10}なら9。

任意消失で固定pairをcollisionさせる最小消失数はw、連続消失ではspanです。
ただし、それらはcross collisionのしきい値であり、完全な量子符号距離を意味しません。


## 5. 手動入力セル

================================  
ここはユーザーが手動入力する  
================================

初期seedは既存解析と同じ `000111 / 010101`。Nは入力せずinflation後に計算します。
window modeでは固定window内の一様m-subsetを用います。連続消失との性能比較も同じwindowに含まれる区間で行い、
全ringのN開始位置で求める理論期待値は別列に保存します。


In [1]:
# ==========================================
# ユーザーが手動入力する解析パラメータ
# ==========================================
SEED_0 = "000111"
SEED_1 = "010101"
PAIR_ID = "L06_W03_P0001"
INFLATION_LIST = [4, 5, 6, 7, 8]
ERASURE_DOMAIN_MODE = "full_ring"  # "full_ring" / "window"
WINDOW_SIZE_LIST = [20, 30, 50]
ERASURE_SIZE_LIST = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
MAX_EXACT_PATTERNS = 100000
NUM_RANDOM_SAMPLES = 10000
RANDOM_SEED = 42
TOP_K_DANGEROUS_SUPPORTS = 20
SAVE_RESULTS = True
SAVE_FIGURES = True

# 追加設定もこのセルだけで変更する。
# Noneなら上の単一pair。複数pairは同じn,N,m,domain内で比較する。
LOGICAL_PAIRS = None
# 例: [{"pair_id":"p0", "seed_0":"00001111", "seed_1":"00011011"}, ...]
WINDOW_START = 0       # windowはこのsiteから右へ。ring上でwrap可能。
LOW_WEIGHT_MAX = 4
LARGE_SPAN_MIN = None  # Noneならceil(LARGE_SPAN_FRACTION*N)
LARGE_SPAN_FRACTION = 0.5
HIGH_RATE_THRESHOLD = 0.8
LOW_RATE_THRESHOLD = 0.2
STORE_PAIR_DETAILS = False  # True: 全cross/self組のshiftとDをcsv.gzに保存
INCLUDE_ALL_SUPPORT_POSITIONS = False  # 全supportはhex maskで保存。危険supportは常に位置も保存。
SAVE_COLLISION_PROFILES = False  # True: 各残存patternのn0(r), n1(r)
COMPUTE_LEAKAGE = False  # True: I_RE, D_Z, D_Xも計算（追加コスト）
MAX_LEAKAGE_DIM = 256   # rho_REを含む最大密行列次元。超過はNA+理由。
MAX_DETAIL_ROWS = 1000000
VALIDATE_SUPPORT_PATTERNS = 3
RUN_FULL_ANALYSIS = False  # benchmark確認後にTrueで全自動計算
OUTPUT_DIR = "results"


================================  
ここから下は基本的に自動計算  
================================

In [2]:
from pathlib import Path
import json, sys, unittest
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from collision_core import *
from collision_analysis import run_analysis, experiment_plan, compare_reference_rates, save_results
from collision_reporting import make_figures, conclusions
pd.set_option("display.max_columns", 24)
CONFIG = {k:v for k,v in list(globals().items()) if k.isupper() and not k.startswith("_") and k != "CONFIG"}
print("inflation: 0→1, 1→10 / site 0: leftmost / self pair: unordered i<j")


inflation: 0→1, 1→10 / site 0: leftmost / self pair: unordered i<j


## 6. Seed条件・実装のvalidation

同じk0/k1、巡回非同値、環の境界を含む1回の隣接01/10交換で相手の巡回同値類に移らないことを自動確認します。
inflation後もorbitの非交差を確認します。原論文の追加的なself-swap条件は、既存のpair間条件とは分けて診断します。

小さい系で文字列の残存pattern判定とbit-mask判定、明示的なpartial traceとKLを照合します。
周期的なorbitの重複除去、M0≠M1、空/全消失、window、期待値と実測平均も検証します。


In [3]:
selected = LOGICAL_PAIRS or [dict(pair_id=PAIR_ID, seed_0=SEED_0, seed_1=SEED_1)]
display(pd.DataFrame([dict(p, **validate_pair(p["seed_0"], p["seed_1"])) for p in selected]))
suite = unittest.defaultTestLoader.loadTestsFromNames(["test_collision_core", "test_collision_kl", "test_collision_analysis"])
test_result = unittest.TextTestRunner(verbosity=1).run(suite)
assert test_result.wasSuccessful(), "validation failed"
print("validation tests:", test_result.testsRun)
assert difference_support("010010", "011000") == {2,4}
assert cyclic_span({5,6,8},20) == 4
assert cyclic_span({2,10},20) == 9


validation tests: 15


...............
----------------------------------------------------------------------
Ran 15 tests in 0.179s

OK


,pair_id,seed_0,seed_1,seed_length,k0,k1,adjacent_swap_valid,paper_no_self_swap_0,paper_no_self_swap_1
0,L06_W03_P0001,000111,010101,6,3,3,True,True,True


## 7–8. Inflation・N・cyclic orbit・計算前benchmark

inflation回数nは0以上。n=0は幾何と実装の検証用であり、原論文のinflation後の保証を適用しません。
orbitは最初に現れた左cyclic shift順で重複を除き、実際のshift indexを保存します。
Python intのXORとbit_countを用い、全消失patternごとのpair XORは行いません。

benchmarkはgeometryの見積りです。KL全pattern計算・漏洩の固有値計算・CSV出力は別のコストがあります。
大規模実験ではcompactなunique support表を標準とし、全pairのDリストはオプションで保存します。


In [4]:
size_rows=[]; benchmark_rows=[]
for p in selected:
    for n in INFLATION_LIST:
        a,b=inflate(p["seed_0"],n),inflate(p["seed_1"],n)
        size_rows.append(dict(pair_id=p["pair_id"],inflation_n=n,N=len(a),orbit_size_0=len(cyclic_orbit(a)),orbit_size_1=len(cyclic_orbit(b))))
        benchmark_rows.append(dict(pair_id=p["pair_id"],**benchmark(p["seed_0"],p["seed_1"],n)))
display(pd.DataFrame(size_rows))
display(pd.DataFrame(benchmark_rows))
plan,skipped=experiment_plan(CONFIG)
display(plan)
print("planned detail rows:", int(plan.num_patterns.sum()) if len(plan) else 0)
if len(skipped): display(skipped)
if SAVE_RESULTS:
    Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
    pd.DataFrame(benchmark_rows).to_csv(Path(OUTPUT_DIR)/"preflight_benchmark.csv",index=False)
    plan.to_csv(Path(OUTPUT_DIR)/"preflight_plan.csv",index=False)


planned detail rows: 564983


,pair_id,inflation_n,N,orbit_size_0,orbit_size_1
0,L06_W03_P0001,4,39,39,13
1,L06_W03_P0001,5,63,63,21
2,L06_W03_P0001,6,102,102,34
3,L06_W03_P0001,7,165,165,55
4,L06_W03_P0001,8,267,267,89


,pair_id,inflation_n,N,orbit_size_0,orbit_size_1,number_cross_pairs,number_self_pairs,sampled_pairs,difference_support_seconds,estimated_total_seconds_naive,estimate_scope
0,L06_W03_P0001,4,39,39,13,507,819,507,0.006724,0.017585,naive feature computation; excludes KL/leakage...
1,L06_W03_P0001,5,63,63,21,1323,2163,1000,0.019997,0.069710,naive feature computation; excludes KL/leakage...
2,L06_W03_P0001,6,102,102,34,3468,5712,1000,0.031305,0.287380,naive feature computation; excludes KL/leakage...
3,L06_W03_P0001,7,165,165,55,9075,15015,1000,0.051908,1.250467,naive feature computation; excludes KL/leakage...
4,L06_W03_P0001,8,267,267,89,23763,39427,1000,0.085518,5.403851,naive feature computation; excludes KL/leakage...


,pair_id,seed_0,seed_1,inflation_n,N,domain_mode,window_size,erasure_size,erasure_type,calculation_mode,population_patterns,num_patterns,status,reason
0,L06_W03_P0001,000111,010101,4,39,full_ring,39,1,arbitrary,exact,39,39,planned,
1,L06_W03_P0001,000111,010101,4,39,full_ring,39,1,contiguous,exact,39,39,planned,
2,L06_W03_P0001,000111,010101,4,39,full_ring,39,2,arbitrary,exact,741,741,planned,
3,L06_W03_P0001,000111,010101,4,39,full_ring,39,2,contiguous,exact,39,39,planned,
4,L06_W03_P0001,000111,010101,4,39,full_ring,39,3,arbitrary,exact,9139,9139,planned,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,L06_W03_P0001,000111,010101,8,267,full_ring,267,8,contiguous,exact,267,267,planned,
96,L06_W03_P0001,000111,010101,8,267,full_ring,267,9,arbitrary,sampled,16582512186174935,10000,planned,
97,L06_W03_P0001,000111,010101,8,267,full_ring,267,9,contiguous,exact,267,267,planned,
98,L06_W03_P0001,000111,010101,8,267,full_ring,267,10,arbitrary,sampled,427828814403313323,10000,planned,


## 9–10. Cross / self pairとdifference support

$$|a_L\rangle=|O_a|^{-1/2}\sum_{x\in O_a}|x\rangle.$$

crossは$O_0\times O_1$の全組。selfは各orbit内の$i<j$のみです。
`support_multiplicity(D)` は同じ絶対site集合を与える組数であり、回転した別のDは別supportです。
同時巡回移動はwとspanを変えないので、feature計算だけに対称性を使います。
全pairの数え方やsupportの絶対位置は変えません。
全supportはhex maskとして保存し、`mask_to_sites(int(mask,16),N)`で絶対位置に戻せます。
危険なsupportには位置リストも保存します。`INCLUDE_ALL_SUPPORT_POSITIONS=True`で全supportの位置も保存します。

`STORE_PAIR_DETAILS=True` の場合、shift・w・span・difference_positions・multiplicityを全組について出力します。
Falseの場合も全組を含む正確なspectrum、unique support表、重要なshift pairを保持します。


In [5]:
RESULTS = None
if RUN_FULL_ANALYSIS:
    RESULTS = run_analysis(CONFIG)
    display(RESULTS["summary"])
else:
    print("準備完了。本計算は手動セルのRUN_FULL_ANALYSIS=Trueで実行。実行済み例は_example.ipynb。")


準備完了。本計算は手動セルのRUN_FULL_ANALYSIS=Trueで実行。実行済み例は_example.ipynb。


## 11–14. Hamming spectrum・span spectrum・joint分布・危険なsupport

`spectra` はpair数で数えたA(w)、B(ell)とunique support数を区別します。
`joint_spectrum` は(w,ell)の同時分布。`dangerous_supports` は小weight、高multiplicityの順に抽出します。
small-weight / large-spanは手動セルの閾値を使用し、pairで重み付けした割合とunique supportの割合を分けます。
小距離が1組だけなのか、多数の組に由来するのかを区別して読んでください。


In [6]:
if RESULTS is not None:
    display(RESULTS["spectra"])
    display(RESULTS["joint_spectrum"].head(30))
    display(RESULTS["dangerous_supports"])
    display(RESULTS["low_weight_large_span_supports"].head(50))


## 15–16. Collision probabilityと期待collision数

全ringの一様m-erasureでは、固定Dのweightがwのとき
$$P(D\subseteq E)=\frac{\binom{N-w}{m-w}}{\binom Nm}\quad(w\le m),$$
それ以外は0。window Wでは$D\subseteq W$を満たすものだけに$N\to |W|$を適用します。

$$\mathbb E[C_{\rm cross}|m]=\sum_D\operatorname{multiplicity}(D)P(D\subseteq E).$$

これは線形性から**組数の期待値として正確**です。$P(C>0)$とは異なり1を超え得ます。
unique support同士でもeventが重なるので、その確率を単純に足しても発生確率にはなりません。
発生確率は全探索または実際の一様抽出で別途集計します。

全ringの連続消失の期待値はN個の開始位置に関する正確な値です。
`expected_cross_collision_contiguous_full_ring` に保存し、window比較にはwindowに含まれる区間の期待値を使います。
0またはN site消失では同じ集合が複数の開始位置から生じても、開始位置を母集団とする定義を保ちます。


In [7]:
if RESULTS is not None:
    display(RESULTS["expected"])


## 17–18. 任意消失と連続消失の比較

任意消失は$\binom{|W|}{m}\le$`MAX_EXACT_PATTERNS`なら全探索。それ以外は**一様な独立復元抽出**です。
重複drawも保持し、同じN/window/mならlogical pair間で同じ消失集合を使います。
sampledの率にはWilson 95%区間、exactは実際の率を保存します。未観測collisionを「存在しない」とは判断しません。

巡回orbitの等重ね合わせは平行移動不変なので、固定mの全ring連続消失は全開始位置で同じKL判定になり、
正確な訂正率は0または1です。固定Dを含む区間の割合は中間値になり得ますが、全cross pairのcollision存在判定とは異なります。
window>Nやm>windowは`skipped`に理由を記録し、windowを黙って縮めません。


In [8]:
if RESULTS is not None:
    display(RESULTS["rates"])
    display(RESULTS["comparison"][["pair_id","inflation_n","N","window_size","erasure_size","d_cross","ell_cross_min","R_KL_arbitrary","R_KL_contiguous","P_cross_collision_arbitrary","diagonal_only_failure_rate_arbitrary","performance_type"]])


## 19. KL条件・既存訂正率との照合・相関

$$\rho_E^{ab}=\operatorname{Tr}_{E^c}|a_L\rangle\langle b_L|.$$

正の等振幅なので`cross_collision == (not KL_offdiagonal)`が成立します。
一方、完全なKLには$\rho_E^{00}=\rho_E^{11}$も必要で、正規化された整数カウントを交差乗算して厳密に判定します。
M0とM1を同一と仮定しません。self collisionがゼロでも、消失部分の周辺分布の差から対角KLが失敗し得ます。
`Delta_C_self=abs(C_self_0-C_self_1)` は診断指標であり、対角条件の代用ではありません。

既存定義の`recoverable_both`は両orbit内の単射性です。量子KL成功とは別の列です。
`1-P_cross_collision`はKL訂正率の上限になりますが、対角だけで失敗する割合を差し引く必要があります。

Pearson・Spearmanは**同じn,N,m,domainで異なるlogical pairが3組以上あり、値が変動する場合のみ**計算します。
単一pairのサイズ系列をpair間相関に混ぜません。少数pairの係数は記述統計であり、汎化予測の検証ではありません。
旧率はdomain・m・Nなどが一致する行だけ比較し、新しい全探索/抽出結果に勝手に置き換えません。


In [9]:
if RESULTS is not None:
    display(RESULTS["correlations"])
    reference_comparison=compare_reference_rates(RESULTS["rates"],Path("reference"))
    display(reference_comparison)
    if SAVE_RESULTS: reference_comparison.to_csv(Path(OUTPUT_DIR)/"reference_rate_comparison.csv",index=False)


## 20. Finite-size scalingとの接続

同じseed pairのnとNに対しd_cross、ell_cross_min、低weight比率、期待collision数を保存します。
固定mで増大するfull ringと、固定W,mのwindowは違う実験です。図表にはdomainを付け、混ぜて結論を出しません。
span/Nの閾値でlow-weight / large-spanを定める場合、Nとともに閾値も変わることを明記します。


In [10]:
if RESULTS is not None:
    display(RESULTS["summary"].sort_values(["pair_id","N"]))


## 21. Information leakageとの接続

以前と同じ$D_Z=\frac12\|\rho_E^{00}-\rho_E^{11}\|_1$、$D_X=\|\rho_E^{01}\|_1$を使用します。
$I(R:E)=1+S(\rho_E)-S(\rho_{RE})$、対数は底2。
`COMPUTE_LEAKAGE=True`で計算し、行列の有効な消失basisだけを使います。
最大行列次元を超える場合、厳密KL・collision計算は保持し、漏洩値のみNAと理由を保存します。NAを0と解釈しません。

`C_cross vs D_X/I_RE`、`Delta_C_self vs D_Z`の相関は同じ設定内のpattern間の記述統計です。
profileは残存patternごとのn0(r),n1(r)ですが、それだけでは消失側の行列要素は復元できません。
KLは消失側のpattern対を含む完全なカウントで判定します。


In [11]:
if RESULTS is not None:
    display(RESULTS["leakage_correlations"])
    display(RESULTS["erasure_detail"].head(15))


## 22–23. DataFrame・主グラフ

主グラフ1–8に加えて、累積A_cross(≤m)/全cross組数とB_cross(≤m)/全cross組数、
訂正率、finite-size、leakageを表示します。scatterは同じ(w,ell)の点を集約し、個数を色/サイズで表します。
保存図はPNG/SVG。図の代表設定はタイトルに明記します。pair間scatterは同じn,N,W,mのgroupを使用し、CSVは全設定を保存します。


In [12]:
if RESULTS is not None:
    FIGURES=make_figures(RESULTS,CONFIG)
    for fig in FIGURES:
        display(fig)
    print("DataFrames:")
    display(pd.DataFrame([{"table":k,"rows":len(v),"columns":len(v.columns)} for k,v in RESULTS.items() if isinstance(v,pd.DataFrame)]))


## 24. 結果まとめ・仮説の判定・次の設計

small weight / large spanは**固定cross pairのcollision機構**として厳密に意味を持ちます。
それが観測された訂正率差の主原因かは、実際のcross発生率と対角だけのKL失敗率を調べて判断します。
両訂正率が一定の場合やpair数不足では「無相関」とは結論しません。

設計候補はd_crossの増大、低weight supportの減少、collision期待数の低減、ell_cross_minの維持です。
同じDのmultiplicityを減らすと期待組数は減りますが、collision発生確率が必ず減るわけではありません。
最終的な設計評価には対角KL一致と、同じdomain/mでの完全なKL訂正率を併用します。

理論背景: [Li & Boyle, Appendix E](https://arxiv.org/html/2311.13040v2#A5)、
[Knill & Laflamme, Theory of quantum error-correcting codes](https://doi.org/10.1103/PhysRevA.55.900)。
今回のseed条件・recoverability・漏洩量の運用上の定義は同梱の既存研究provenanceに従います。


In [13]:
if RESULTS is not None:
    final_text=conclusions(RESULTS)
    display(Markdown(final_text))
    display(RESULTS["summary"])
    display(RESULTS["spectra"][RESULTS["spectra"].metric_type.isin(["cross_hamming","cross_span"])])
    display(RESULTS["dangerous_supports"])
    display(RESULTS["comparison"][["pair_id","inflation_n","N","erasure_size","expected_cross_collision_arbitrary","expected_cross_collision_contiguous","R_KL_arbitrary","R_KL_contiguous"]])
    display(RESULTS["correlations"])
    if SAVE_RESULTS: (Path(OUTPUT_DIR)/"conclusions.md").write_text(final_text,encoding="utf-8")
else:
    print("本番の結果は未計算です。準備表と実行済み小規模例を確認してください。")


本番の結果は未計算です。準備表と実行済み小規模例を確認してください。
